# Camada Silver — Limpeza, Tipos e Padronização

Transforma os dados raw da camada Bronze aplicando:
- **Conversão de tipos**: strings → INT, DECIMAL, TIMESTAMP
- **Tratamento de nulos**: espaços " " → NULL em colunas numéricas
- **Padronização**: trimestres ("1º" → 1), índices ("58,84" → 58.84)
- **Deduplicação**: ROW_NUMBER() mantém o registro mais recente

## Tabelas Silver

| Tabela | Origem Bronze | Dedupla por |
| --- | --- | --- |
| `reclamacoes_bacen` | `reclamacoes_bacen` | ano + trimestre + instituição |
| `ouvidorias_bacen` | `ouvidorias_bacen` | ano + período + tipo + instituição |
| `consorcios_bacen` | `consorcios_bacen` | ano + semestre + administradora |

> **Pré-requisito**: execute `01_bronze_bacen` antes deste notebook.

## 📋 Resumo da Camada Silver — Objetivo do MVP

### Por que foi feita

Os dados extraídos do portal do Bacen chegam à **camada Bronze** em formato bruto: todas as colunas como texto (`STRING`), com sujeira típica de extrações tabulares — espaços vazios em vez de `NULL`, separadores de milhar e vírgula decimal em valores numéricos, trimestres e semestres rotulados como `"1º"`, `"2º"` e registros duplicados decorrentes de múltiplas cargas históricas.

Esses dados, no estado em que se encontram, **não são adequados para análise nem modelagem**. Qualquer cálculo de média, soma ou comparação exigiria limpeza repetida em cada consulta, tornando os dashboards frágeis e inconsistentes. A camada Silver resolve esse problema transformando os dados uma única vez, de forma **reprodutível e versionada**.

### O que foi feito

Para cada uma das três fontes (Reclamações, Ouvidorias e Consórcios), aplicou-se um pipeline de transformação com quatro etapas:

| Etapa | Descrição |
| --- | --- |
| **Conversão de tipos** | `CAST` de strings para `INT`, `LONG`, `DECIMAL` e `TIMESTAMP`, garantindo que métricas sejam numericamente operáveis |
| **Tratamento de nulos** | `NULLIF(TRIM(coluna), '')` converte espaços e strings vazias em `NULL` real, evitando falhas em agregações |
| **Padronização** | `REPLACE` de separadores de milhar (`.`) e vírgula decimal (`,` → `.`) em índices e notas; `SUBSTRING` para extrair o número de trimestre/semestre (`"1º"` → `1`) |
| **Deduplicação** | `ROW_NUMBER()` particionado pela chave de negócio de cada tabela, ordenado por `data_carga DESC`, mantendo apenas o registro mais recente |

### Resultado

Três tabelas **tipadas, limpas e sem duplicatas**, prontas para consumo pela camada Gold — onde ocorrem as agregações finais e a construção de indicadores de negócio:

- `workspace.bacen_silver.reclamacoes_bacen`
- `workspace.bacen_silver.ouvidorias_bacen`
- `workspace.bacen_silver.consorcios_bacen`

> **Valor para o MVP:** a camada Silver atua como a *fonte única de verdade* intermediária. Ao centralizar a limpeza aqui, garantimos que qualquer análise, dashboard ou modelo construído sobre esses dados parta de uma base consistente — reduzindo retrabalho e eliminando inconsistências entre consultas.



## 1. Reclamações de Clientes

Limpeza e padronização dos dados de reclamações: conversão de tipos, tratamento de nulos, padronização de trimestres e índices (vírgula → ponto decimal). Deduplicação por (ano, trimestre, instituição).

In [0]:
%sql
-- ============================================================================
-- CAMADA SILVER - LIMPEZA, TIPOS E PADRONIZAÇÃO (CTAS)
-- ============================================================================
-- Cria a tabela Silver de Reclamações a partir da Bronze raw. Aplica:
--   • CAST de strings para INT/DECIMAL/TIMESTAMP em todas as colunas
--   • NULLIF(TRIM(...), '') para converter espaços vazios em NULL real
--   • REPLACE de separadores de milhar e vírgula decimal no índice
--   • SUBSTRING para extrair o número do trimestre ("1º" → 1)
--   • ROW_NUMBER() particionado por (ano, trimestre, instituição) para
--     eliminar duplicatas mantendo o registro de data_carga mais recente
-- Resultado: tabela tipada, sem duplicatas, pronta para a camada Gold.
-- ============================================================================

CREATE OR REPLACE TABLE workspace.bacen_silver.reclamacoes_bacen AS
WITH bronze_clean AS (
  SELECT
    -- Dimensões temporais
    CAST(ano AS INT)                                       AS ano,
    CAST(SUBSTRING(trimestre, 1, 1) AS INT)               AS trimestre,
    -- Dimensões categóricas
    TRIM(categoria)                                        AS categoria,
    TRIM(tipo)                                             AS tipo,
    NULLIF(TRIM(cnpj_if), '')                              AS cnpj_if,
    TRIM(instituicao_financeira)                           AS instituicao_financeira,
    -- Métricas (índice: remove separador de milhar '.', troca vírgula por ponto decimal)
    CAST(REPLACE(REPLACE(NULLIF(TRIM(indice), ''), '.', ''), ',', '.')
         AS DECIMAL(10, 2))                               AS indice,
    CAST(NULLIF(TRIM(quantidade_de_reclamacoes_reguladas_procedentes), '') AS LONG)  AS qtd_reclamacoes_reguladas_procedentes,
    CAST(NULLIF(TRIM(quantidade_de_reclamacoes_reguladas_outras), '')        AS LONG)  AS qtd_reclamacoes_reguladas_outras,
    CAST(NULLIF(TRIM(quantidade_de_reclamacoes_nao_reguladas), '')           AS LONG)  AS qtd_reclamacoes_nao_reguladas,
    CAST(NULLIF(TRIM(quantidade_total_de_reclamacoes), '')                   AS LONG)  AS qtd_total_reclamacoes,
    CAST(NULLIF(TRIM(quantidade_total_de_clientes_ccs_e_scr), '')            AS LONG)  AS qtd_total_clientes_ccs_scr,
    CAST(NULLIF(TRIM(quantidade_de_clientes_ccs), '')                       AS LONG)  AS qtd_clientes_ccs,
    CAST(NULLIF(TRIM(quantidade_de_clientes_scr), '')                       AS LONG)  AS qtd_clientes_scr,
    CAST(NULLIF(TRIM(quantidade_de_clientes_fgc), '')                       AS LONG)  AS qtd_clientes_fgc,
    -- Colunas extras (2024+)
    CAST(NULLIF(TRIM(quantidade_total_de_reclamacoes_respondidas), '')                  AS LONG) AS qtd_total_reclamacoes_respondidas,
    CAST(NULLIF(TRIM(quantidade_de_reclamacoes_reguladas_procedentes_extrapoladas), '') AS LONG) AS qtd_reclamacoes_reguladas_proc_extrapoladas,
    CAST(NULLIF(TRIM(quantidade_de_reclamacoes_procedentes), '')                         AS LONG) AS qtd_reclamacoes_procedentes,
    CAST(NULLIF(TRIM(quantidade_de_reclamacoes_procedentes_extrapoladas), '')            AS LONG) AS qtd_reclamacoes_procedentes_extrapoladas,
    CAST(NULLIF(TRIM(quantidade_total_de_reclamacoes_analisadas), '')                    AS LONG) AS qtd_total_reclamacoes_analisadas,
    -- Metadados de carga
    CAST(ano_extracao AS INT)                              AS ano_extracao,
    CAST(periodo_extracao AS INT)                          AS periodo_extracao,
    TRIM(tipo_instituicao)                                 AS tipo_instituicao,
    CAST(data_carga AS TIMESTAMP)                          AS data_carga
  FROM workspace.bacen_bronze.reclamacoes_bacen
),
dedup AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY ano, trimestre, instituicao_financeira
      ORDER BY data_carga DESC
    ) AS _rn
  FROM bronze_clean
)
SELECT * EXCEPT (_rn)
FROM dedup
WHERE _rn = 1

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- ============================================================================
-- VERIFICAÇÃO DA CAMADA SILVER — RECLAMAÇÕES
-- ============================================================================
-- Valida a tabela Silver recém-criada com cinco consultas:
--   1. DESCRIBE — schema completo com tipos convertidos
--   2. COUNT + DISTINCT — total de registros e chaves únicas
--   3. GROUP BY ano_extracao, periodo_extracao — distribuição por período
--   4. Amostra de 10 linhas com indice não-nulo para inspeção visual
--   5. Soma de NULLs por coluna — confirma que espaços viraram NULL
-- ============================================================================

-- Schema e contagem
DESCRIBE workspace.bacen_silver.reclamacoes_bacen;

-- Total de registros e colunas
SELECT COUNT(*) AS total_registros, COUNT(DISTINCT CONCAT(ano, '-', trimestre, '-', instituicao_financeira)) AS chaves_unicas
FROM workspace.bacen_silver.reclamacoes_bacen;

-- Distribuição por ano e trimestre
SELECT ano_extracao, periodo_extracao, COUNT(*) AS registros
FROM workspace.bacen_silver.reclamacoes_bacen
GROUP BY ano_extracao, periodo_extracao
ORDER BY ano_extracao, periodo_extracao;

-- Amostra: verificar tipos limpos (índice decimal, trimestre como INT, nulos reais)
SELECT ano, trimestre, tipo, instituicao_financeira, indice,
       qtd_total_reclamacoes, qtd_clientes_ccs, qtd_clientes_scr, qtd_clientes_fgc
FROM workspace.bacen_silver.reclamacoes_bacen
WHERE indice IS NOT NULL
LIMIT 10;

-- Verificar: espaços viraram NULL?
SELECT
  SUM(CASE WHEN qtd_clientes_ccs IS NULL THEN 1 ELSE 0 END) AS ccs_nulls,
  SUM(CASE WHEN qtd_clientes_scr IS NULL THEN 1 ELSE 0 END) AS scr_nulls,
  SUM(CASE WHEN qtd_clientes_fgc IS NULL THEN 1 ELSE 0 END) AS fgc_nulls,
  SUM(CASE WHEN cnpj_if IS NULL THEN 1 ELSE 0 END) AS cnpj_nulls,
  SUM(CASE WHEN indice IS NULL THEN 1 ELSE 0 END) AS indice_nulls
FROM workspace.bacen_silver.reclamacoes_bacen;

ccs_nulls,scr_nulls,fgc_nulls,cnpj_nulls,indice_nulls
349,242,4380,3150,3504


## 2. Ouvidorias

Limpeza e tipagem dos dados de ouvidorias: notas como DECIMAL, tempo médio como DECIMAL, deduplicação por (ano, período, tipo_período, instituição).

In [0]:
%sql
-- ============================================================================
-- CAMADA SILVER OUVIDORIAS - LIMPEZA E TIPOS (CTAS)
-- ============================================================================
-- Cria a tabela Silver de Ouvidorias a partir da Bronze raw. Aplica:
--   • CAST de strings para INT (contagens) e DECIMAL (notas e tempo médio)
--   • REPLACE de vírgula decimal por ponto em nota_prazo, nota_qualidade,
--     nota_final e tempo_medio_respostas
--   • NULLIF(TRIM(...), '') em cnpj_if para espaços vazios
--   • ROW_NUMBER() particionado por (ano, periodo, tipo_periodo,
--     instituição) para eliminar duplicatas mantendo o registro mais recente
-- Resultado: tabela tipada com notas decimais, pronta para a camada Gold.
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_silver.ouvidorias_bacen AS
WITH bronze_clean AS (
  SELECT
    CAST(ano AS INT) AS ano, CAST(periodo AS INT) AS periodo,
    TRIM(tipo_periodo) AS tipo_periodo, TRIM(categoria) AS categoria, TRIM(tipo) AS tipo,
    NULLIF(TRIM(cnpj_if), '') AS cnpj_if, TRIM(instituicao_financeira) AS instituicao_financeira,
    CAST(num_respostas_fornecidas AS INT) AS num_respostas_fornecidas,
    CAST(num_respostas_atrasadas AS INT) AS num_respostas_atrasadas,
    CAST(REPLACE(tempo_medio_respostas, ',', '.') AS DECIMAL(10, 6)) AS tempo_medio_respostas,
    CAST(qtd_reclamacoes_proc_qualid AS INT) AS qtd_reclamacoes_proc_qualid,
    CAST(qtd_reclamacoes_encerradas AS INT) AS qtd_reclamacoes_encerradas,
    CAST(qtd_reclamacoes_proc_ouvidoria AS INT) AS qtd_reclamacoes_proc_ouvidoria,
    TRIM(adesao_consumidor_gov) AS adesao_consumidor_gov,
    CAST(REPLACE(nota_prazo, ',', '.') AS DECIMAL(10, 6)) AS nota_prazo,
    CAST(REPLACE(nota_qualidade, ',', '.') AS DECIMAL(10, 6)) AS nota_qualidade,
    CAST(REPLACE(nota_final, ',', '.') AS DECIMAL(10, 6)) AS nota_final,
    CAST(data_carga AS TIMESTAMP) AS data_carga
  FROM workspace.bacen_bronze.ouvidorias_bacen
),
dedup AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY ano, periodo, tipo_periodo, instituicao_financeira ORDER BY data_carga DESC) AS _rn
  FROM bronze_clean
)
SELECT * EXCEPT (_rn) FROM dedup WHERE _rn = 1

num_affected_rows,num_inserted_rows


## 3. Consórcios

Limpeza e tipagem dos dados de consórcios: índice decimal, semestre como INT, deduplicação por (ano, semestre, administradora).

In [0]:
%sql
-- ============================================================================
-- CAMADA SILVER CONSÓRCIOS - LIMPEZA E TIPOS (CTAS)
-- ============================================================================
-- Cria a tabela Silver de Consórcios a partir da Bronze raw. Aplica:
--   • CAST de strings para INT (semestre, contagens) e DECIMAL (índice)
--   • REPLACE de separadores de milhar e vírgula decimal no índice
--   • SUBSTRING para extrair o número do semestre ("1º" → 1)
--   • NULLIF(TRIM(...), '') em cnpj_ac e colunas numéricas
--   • ROW_NUMBER() particionado por (ano, semestre, administradora) para
--     eliminar duplicatas mantendo o registro de data_carga mais recente
-- Resultado: tabela tipada, sem duplicatas, pronta para a camada Gold.
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_silver.consorcios_bacen AS
WITH bronze_clean AS (
  SELECT
    CAST(ano AS INT) AS ano,
    CAST(SUBSTRING(semestre, 1, 1) AS INT) AS semestre,
    NULLIF(TRIM(cnpj_ac), '') AS cnpj_ac,
    TRIM(administradora_de_consorcio) AS administradora,
    CAST(REPLACE(REPLACE(NULLIF(TRIM(indice), ''), '.', ''), ',', '.') AS DECIMAL(10, 2)) AS indice,
    CAST(NULLIF(TRIM(quantidade_de_reclamacoes_reguladas_procedentes), '') AS LONG) AS qtd_reclamacoes_procedentes,
    CAST(NULLIF(TRIM(quantidade_de_reclamacoes_reguladas_outras), '') AS LONG) AS qtd_reclamacoes_outras,
    CAST(NULLIF(TRIM(quantidade_de_reclamacoes_nao_reguladas), '') AS LONG) AS qtd_reclamacoes_nao_reguladas,
    CAST(NULLIF(TRIM(quantidade_total_de_reclamacoes), '') AS LONG) AS qtd_total_reclamacoes,
    CAST(NULLIF(TRIM(quantidade_de_clientes_consorciados), '') AS LONG) AS qtd_clientes_consorciados,
    CAST(_ano_extracao AS INT) AS ano_extracao,
    CAST(_periodo_extracao AS INT) AS periodo_extracao,
    CAST(_data_carga AS TIMESTAMP) AS data_carga
  FROM workspace.bacen_bronze.consorcios_bacen
),
dedup AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY ano, semestre, administradora ORDER BY data_carga DESC) AS _rn
  FROM bronze_clean
)
SELECT * EXCEPT (_rn) FROM dedup WHERE _rn = 1

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- ============================================================================
-- VERIFICAÇÃO FINAL — TODAS AS TABELAS SILVER
-- ============================================================================
-- Conta os registros de cada tabela Silver criada neste notebook
-- (reclamacoes_bacen, ouvidorias_bacen, consorcios_bacen) em uma
-- única consulta UNION ALL. Serve como checagem rápida para confirmar
-- que as três fontes foram transformadas e não estão vazias.
-- ============================================================================
SELECT 'reclamacoes_bacen' AS tabela, COUNT(*) AS registros FROM workspace.bacen_silver.reclamacoes_bacen
UNION ALL SELECT 'ouvidorias_bacen', COUNT(*) FROM workspace.bacen_silver.ouvidorias_bacen
UNION ALL SELECT 'consorcios_bacen', COUNT(*) FROM workspace.bacen_silver.consorcios_bacen
ORDER BY tabela

tabela,registros
consorcios_bacen,2015
ouvidorias_bacen,3233
reclamacoes_bacen,5060
